In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/data.csv")

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


This notebook demonstrates the feature engineering pipeline used to transform raw transaction data into a model-ready dataset.

In [2]:
agg_features = df.groupby("CustomerId").agg(
    TotalTransactionAmount=("Amount","sum"),
    AverageTransactionAmount=("Amount","mean"),
    TransactionCount=("TransactionId","count"),
    TransactionStd=("Amount","std")
).reset_index()

agg_features.head()

,CustomerId,TotalTransactionAmount,AverageTransactionAmount,TransactionCount,TransactionStd
0,CustomerId_1,-10000.0,-10000.000000,1,NaN
1,CustomerId_10,-10000.0,-10000.000000,1,NaN
2,CustomerId_1001,20000.0,4000.000000,5,6558.963333
3,CustomerId_1002,4225.0,384.090909,11,560.498966
4,CustomerId_1003,20000.0,3333.333333,6,6030.478146


In [3]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

In [4]:
df["TransactionHour"] = df["TransactionStartTime"].dt.hour

df["TransactionDay"] = df["TransactionStartTime"].dt.day

df["TransactionMonth"] = df["TransactionStartTime"].dt.month

df["TransactionYear"] = df["TransactionStartTime"].dt.year

In [5]:
df[
    [
        "TransactionHour",
        "TransactionDay",
        "TransactionMonth",
        "TransactionYear"
    ]
].head()

,TransactionHour,TransactionDay,TransactionMonth,TransactionYear
0,2,15,11,2018
1,2,15,11,2018
2,2,15,11,2018
3,3,15,11,2018
4,3,15,11,2018


Temporal features help capture transaction patterns and customer behavior over time.

In [6]:
df.isnull().sum()

TransactionId           0
BatchId                 0
AccountId               0
SubscriptionId          0
CustomerId              0
CurrencyCode            0
CountryCode             0
ProviderId              0
ProductId               0
ProductCategory         0
ChannelId               0
Amount                  0
Value                   0
TransactionStartTime    0
PricingStrategy         0
FraudResult             0
TransactionHour         0
TransactionDay          0
TransactionMonth        0
TransactionYear         0
dtype: int64

In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="most_frequent")

Missing values were handled using imputation to prevent information loss.

In [8]:
categorical_cols = [
    "ProductCategory",
    "ChannelId",
    "ProviderId"
]

In [9]:
pd.get_dummies(
    df[categorical_cols]
).head()

,ProductCategory_airtime,ProductCategory_data_bundles,ProductCategory_financial_services,ProductCategory_movies,ProductCategory_other,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_1,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5,ProviderId_ProviderId_1,ProviderId_ProviderId_2,ProviderId_ProviderId_3,ProviderId_ProviderId_4,ProviderId_ProviderId_5,ProviderId_ProviderId_6
0,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
1,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False
2,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False
4,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [14]:
numerical_cols = [
    "Amount",
    "Value",
    "TransactionHour",
    "TransactionDay",
    "TransactionMonth",
    "TransactionYear"
]
categorical_cols = [
    "CurrencyCode",
    "ProviderId",
    "ProductId",
    "ProductCategory",
    "ChannelId",
    "PricingStrategy"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

In [15]:
print(df.columns.tolist())

['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult', 'TransactionHour', 'TransactionDay', 'TransactionMonth', 'TransactionYear']


In [16]:
processed_data = preprocessor.fit_transform(df)

processed_data.shape

(95662, 53)

In [18]:
processed_data = preprocessor.fit_transform(df)

In [19]:
import pandas as pd
import os

os.makedirs("../data/processed", exist_ok=True)

processed_df = pd.DataFrame(
    processed_data.toarray()
    if hasattr(processed_data, "toarray")
    else processed_data
)

processed_df.to_csv(
    "../data/processed/processed_data.csv",
    index=False
)

print("Processed data saved successfully!")

Processed data saved successfully!


In [20]:
processed_data = preprocessor.fit_transform(df)

processed_data.shape

(95662, 53)

In [21]:
processed_df = pd.DataFrame(
    processed_data.toarray()
    if hasattr(processed_data, "toarray")
    else processed_data
)

processed_df.to_csv(
    "../data/processed/processed_data.csv",
    index=False
)

Feature Engineering Summary

✓ Aggregate customer features created

✓ Time-based features extracted

✓ Missing values handled

✓ Categorical variables encoded

✓ Numerical features standardized

✓ Reproducible sklearn Pipeline implemented

The resulting dataset is ready for RFM analysis and proxy target variable creation.